# Joint-training reference

This notebook analyzes the separate all-data-at-once reference. Each adapter sees all 2,000 Helpful, 2,000 Safe, and 2,000 Quality training pairs in one globally shuffled pass. There is no task order, so joint training has final scores but **does not have a forgetting score**.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
candidates = [here, *here.parents]
REPO = next((p for p in candidates if (p / 'configs/experiment_protocol.json').exists()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the mfr-dpo repository')
if 'MFR_OUTPUT_DIR' not in os.environ:
    raise EnvironmentError('Set MFR_OUTPUT_DIR to the absolute CARC artifact directory first')
ARTIFACT_DIR = Path(os.environ['MFR_OUTPUT_DIR']).expanduser().resolve()
DATASETS = ['helpful', 'safe', 'quality']
LABEL = {'helpful':'Helpful', 'safe':'Safe', 'quality':'Quality'}
COLORS = {'helpful':'#367bd2', 'safe':'#d1495b', 'quality':'#22a879'}
print('Repository:', REPO)
print('Artifacts:', ARTIFACT_DIR)

## 1. Check that all five runs finished

The expected seeds are 0, 1, 2, 3, and 4. Incomplete folders are shown but excluded from the score tables.

In [ ]:
inventory, frames = [], []
for folder in sorted((ARTIFACT_DIR / 'joint_runs').glob('v2_joint_s*')):
    settings_path = folder / 'settings.json'
    results_path = folder / 'results.csv'
    if not settings_path.exists():
        continue
    settings = json.loads(settings_path.read_text())
    complete = (folder / 'COMPLETE.json').exists()
    inventory.append({'Seed': settings['seed'], 'Run': settings['run_name'],
                      'Complete': complete, 'Training pairs': settings['total_train_pairs'],
                      'Optimizer steps': settings['optimizer_steps']})
    if complete and results_path.exists():
        frame = pd.read_csv(results_path)
        frame['folder'] = str(folder)
        frames.append(frame)
inventory = pd.DataFrame(inventory).sort_values('Seed') if inventory else pd.DataFrame()
print(inventory.to_string(index=False) if len(inventory) else 'No joint runs found.')
if len(inventory) != 5 or not inventory['Complete'].all():
    print('WARNING: the five-seed joint grid is not complete yet.')
joint = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 2. Final validation scores for every seed

These are preference-pair accuracies after the single joint-training pass. The Average column gives equal weight to the three behaviors.

In [ ]:
if joint.empty:
    print('No completed joint results to analyze.')
else:
    final = joint[joint.stage == 1].copy()
    per_seed = (final.pivot(index='seed', columns='eval_set', values='accuracy')
                .reindex(columns=DATASETS))
    per_seed['average'] = per_seed.mean(axis=1)
    shown = per_seed.rename(columns={**LABEL, 'average':'Average'}).reset_index().rename(columns={'seed':'Seed'})
    print(shown.round(2).to_string(index=False))

## 3. Mean and variation across all five seeds

Every graph includes Helpful, Safe, and Quality. Error bars are one standard deviation across seeds; they describe run-to-run variation rather than a confidence interval.

In [ ]:
if not joint.empty:
    seed_std = lambda values: values.std(ddof=1) if len(values) > 1 else 0.0
    aggregate = pd.DataFrame({
        'Dataset': [LABEL[d] for d in DATASETS] + ['Three-behavior average'],
        'Mean accuracy': [per_seed[d].mean() for d in DATASETS] + [per_seed['average'].mean()],
        'Seed standard deviation': [seed_std(per_seed[d]) for d in DATASETS] + [seed_std(per_seed['average'])],
    })
    print(aggregate.round(2).to_string(index=False))
    means = [per_seed[d].mean() for d in DATASETS]
    errors = [seed_std(per_seed[d]) for d in DATASETS]
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.bar([LABEL[d] for d in DATASETS], means, yerr=errors, capsize=5,
           color=[COLORS[d] for d in DATASETS])
    ax.set_ylabel('Final preference accuracy (%)')
    ax.set_title('Joint training: final validation accuracy across Seeds 0–4')
    ax.grid(axis='y', alpha=.25)
    plt.tight_layout()
    plt.show()

## How to interpret this reference

Joint training shows performance when old data never becomes unavailable. Compare its final per-behavior and average scores with the continual methods, but do not say that joint training prevents forgetting: it avoids the sequential problem entirely. It is also not guaranteed to win, because mixing competing preferences can create interference even when all data are present.